In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Set paths and assumptions
# --------------------------------------------------
PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "inventory"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "retail_sales_final.csv"

# These are planning assumptions, not values supplied by the dataset.
HISTORY_DAYS = 90
ASSUMED_LEAD_TIME_DAYS = 14
ASSUMED_REVIEW_PERIOD_DAYS = 7
SAFETY_FACTOR_Z = 1.65  # Approximate 95% one-sided safety factor

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Master dataset not found: {INPUT_PATH}")

# Load only the nine project-analysis columns.
# The three helper flags are deliberately excluded.
analysis_columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country", "Revenue"
]

df = pd.read_csv(
    INPUT_PATH,
    dtype={"StockCode": "string"},
    low_memory=False
)

missing_columns = [c for c in analysis_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

sales = df[analysis_columns].copy()

sales["InvoiceDate"] = pd.to_datetime(
    sales["InvoiceDate"], errors="coerce"
)
sales["Quantity"] = pd.to_numeric(sales["Quantity"], errors="coerce")
sales["Price"] = pd.to_numeric(sales["Price"], errors="coerce")

# Use eligible purchases; keep the master CSV unchanged.
sales = sales[
    sales["InvoiceDate"].notna()
    & sales["StockCode"].notna()
    & (sales["Quantity"] > 0)
    & (sales["Price"] > 0)
].copy()

sales["Date"] = sales["InvoiceDate"].dt.normalize()
sales["Sales_Revenue"] = sales["Quantity"] * sales["Price"]

if sales.empty:
    raise ValueError("No eligible product sales were found.")

as_of_date = sales["Date"].max()
window_start = as_of_date - pd.Timedelta(days=HISTORY_DAYS - 1)
calendar_dates = pd.date_range(
    start=window_start,
    end=as_of_date,
    freq="D"
)

recent = sales[
    sales["Date"].between(window_start, as_of_date)
].copy()

if recent.empty:
    raise ValueError("No eligible sales found in the 90-day window.")

# --------------------------------------------------
# 2. Aggregate product sales by calendar day
# Include zero-demand days when calculating variability.
# --------------------------------------------------
daily_units = recent.pivot_table(
    index="Date",
    columns="StockCode",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

daily_units = daily_units.reindex(
    index=calendar_dates,
    fill_value=0
).fillna(0)

daily_revenue = recent.pivot_table(
    index="Date",
    columns="StockCode",
    values="Sales_Revenue",
    aggfunc="sum",
    fill_value=0
)

daily_revenue = daily_revenue.reindex(
    index=calendar_dates,
    columns=daily_units.columns,
    fill_value=0
).fillna(0)

mean_daily_units = daily_units.mean(axis=0)
std_daily_units = daily_units.std(axis=0, ddof=1).fillna(0)
mean_daily_revenue = daily_revenue.mean(axis=0)

# Include products with at least one eligible sale in this 90-day window.
active_codes = mean_daily_units[mean_daily_units > 0].index

demand = pd.DataFrame({
    "StockCode": active_codes.astype(str),
    "Average_Daily_Units": mean_daily_units.loc[active_codes].values,
    "Daily_Units_Std": std_daily_units.loc[active_codes].values,
    "Average_Daily_Revenue": mean_daily_revenue.loc[active_codes].values
})

# Representative description: most common observed description in the window.
description_map = (
    recent.groupby("StockCode")["Description"]
    .agg(
        lambda values: (
            values.dropna().mode().iloc[0]
            if not values.dropna().mode().empty
            else "UNKNOWN"
        )
    )
)

demand["Description"] = demand["StockCode"].map(description_map).fillna("UNKNOWN")

# --------------------------------------------------
# 3. Estimate demand, safety stock and reorder points
# --------------------------------------------------
demand["Forecast_30Day_Demand_Units"] = (
    demand["Average_Daily_Units"] * 30
)

demand["Forecast_30Day_Revenue"] = (
    demand["Average_Daily_Revenue"] * 30
)

demand["Assumed_Lead_Time_Days"] = ASSUMED_LEAD_TIME_DAYS
demand["Assumed_Review_Period_Days"] = ASSUMED_REVIEW_PERIOD_DAYS
demand["Safety_Factor_Z"] = SAFETY_FACTOR_Z

demand["Safety_Stock_Units"] = np.ceil(
    SAFETY_FACTOR_Z
    * demand["Daily_Units_Std"]
    * np.sqrt(ASSUMED_LEAD_TIME_DAYS)
)

demand["Estimated_Reorder_Point_Units"] = np.ceil(
    demand["Average_Daily_Units"] * ASSUMED_LEAD_TIME_DAYS
    + demand["Safety_Stock_Units"]
)

coverage_days = ASSUMED_LEAD_TIME_DAYS + ASSUMED_REVIEW_PERIOD_DAYS

demand["Estimated_Target_Stock_Units"] = np.ceil(
    demand["Average_Daily_Units"] * coverage_days
    + SAFETY_FACTOR_Z
    * demand["Daily_Units_Std"]
    * np.sqrt(coverage_days)
)

# Stock on hand and supplier lead time were not supplied.
# Leave real-world reorder quantities blank rather than inventing them.
demand["Current_Stock_On_Hand"] = pd.NA
demand["Actual_Supplier_Lead_Time_Days"] = pd.NA
demand["Suggested_Reorder_Quantity"] = pd.NA
demand["Reorder_Status"] = "Pending actual stock and lead-time data"

demand["Historical_Window_Start"] = window_start.date().isoformat()
demand["Historical_Window_End"] = as_of_date.date().isoformat()
demand["Historical_Window_Days"] = HISTORY_DAYS

# Put the most important fields first.
output_columns = [
    "StockCode",
    "Description",
    "Historical_Window_Start",
    "Historical_Window_End",
    "Historical_Window_Days",
    "Average_Daily_Units",
    "Daily_Units_Std",
    "Forecast_30Day_Demand_Units",
    "Average_Daily_Revenue",
    "Forecast_30Day_Revenue",
    "Assumed_Lead_Time_Days",
    "Actual_Supplier_Lead_Time_Days",
    "Assumed_Review_Period_Days",
    "Safety_Factor_Z",
    "Safety_Stock_Units",
    "Estimated_Reorder_Point_Units",
    "Estimated_Target_Stock_Units",
    "Current_Stock_On_Hand",
    "Suggested_Reorder_Quantity",
    "Reorder_Status"
]

demand = demand[output_columns].sort_values(
    "Forecast_30Day_Demand_Units",
    ascending=False
).reset_index(drop=True)

# --------------------------------------------------
# 4. Save baseline and a template for real stock inputs
# --------------------------------------------------
baseline_path = REPORT_DIR / "product_demand_reorder_baseline.csv"
stock_template_path = PROCESSED_DIR / "inventory_stock_input_template.csv"
summary_path = REPORT_DIR / "inventory_baseline_summary.csv"

demand.to_csv(baseline_path, index=False)

# Separate practical template: fill in these fields from real inventory records.
stock_template = demand[[
    "StockCode",
    "Description",
    "Current_Stock_On_Hand",
    "Actual_Supplier_Lead_Time_Days"
]].copy()

stock_template["Current_Stock_Captured_On"] = pd.NA
stock_template.to_csv(stock_template_path, index=False)

summary = pd.DataFrame([
    {"Metric": "Historical window start", "Value": window_start.date().isoformat()},
    {"Metric": "Historical window end", "Value": as_of_date.date().isoformat()},
    {"Metric": "Calendar days analysed", "Value": len(calendar_dates)},
    {"Metric": "Active products in recent window", "Value": len(demand)},
    {
        "Metric": "Total eligible revenue in window",
        "Value": round(recent["Sales_Revenue"].sum(), 2)
    },
    {
        "Metric": "Total eligible units in window",
        "Value": round(recent["Quantity"].sum(), 2)
    },
    {
        "Metric": "Assumed lead time (days)",
        "Value": ASSUMED_LEAD_TIME_DAYS
    },
    {
        "Metric": "Assumed review period (days)",
        "Value": ASSUMED_REVIEW_PERIOD_DAYS
    },
    {
        "Metric": "Stock-on-hand values supplied",
        "Value": 0
    },
    {
        "Metric": "Actual supplier lead times supplied",
        "Value": 0
    }
])

summary.to_csv(summary_path, index=False)

print("=" * 70)
print("RETAILPULSE - INVENTORY DEMAND BASELINE")
print("=" * 70)
print(f"Historical window: {window_start.date()} to {as_of_date.date()}")
print(f"Calendar days analysed: {len(calendar_dates)}")
print(f"Active products: {len(demand):,}")
print(f"Eligible sales revenue in window: {recent['Sales_Revenue'].sum():,.2f}")
print(f"Eligible units in window: {recent['Quantity'].sum():,.2f}")

print("\nTop 10 products by estimated 30-day unit demand:")
display(
    demand[[
        "StockCode",
        "Description",
        "Average_Daily_Units",
        "Forecast_30Day_Demand_Units",
        "Safety_Stock_Units",
        "Estimated_Reorder_Point_Units",
        "Estimated_Target_Stock_Units"
    ]].head(10).round(2)
)

print("\nSummary:")
display(summary)

print("\nSaved files:")
print(baseline_path)
print(stock_template_path)
print(summary_path)
print("\nThe master dataset was not modified.")


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "inventory"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = PROCESSED_DIR / "retail_sales_final.csv"
BASELINE_PATH = REPORT_DIR / "product_demand_reorder_baseline.csv"

# Load only the nine analysis columns; ignore the three helper flags.
columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country", "Revenue"
]

df = pd.read_csv(
    MASTER_PATH,
    usecols=columns,
    dtype={"StockCode": "string"},
    low_memory=False
)
baseline = pd.read_csv(BASELINE_PATH, dtype={"StockCode": "string"})

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Date"] = df["InvoiceDate"].dt.normalize()

df = df[
    df["Date"].notna()
    & df["StockCode"].notna()
    & (df["Quantity"] > 0)
    & (df["Price"] > 0)
].copy()

# Match the same 90-day window used in the inventory baseline.
end_date = df["Date"].max()
start_date = end_date - pd.Timedelta(days=89)
recent = df[df["Date"].between(start_date, end_date)].copy()

# Identify the top 10 products from the saved baseline.
top10 = baseline.nlargest(10, "Forecast_30Day_Demand_Units").copy()
top_codes = top10["StockCode"].astype(str).tolist()

# Aggregate quantity by product and day.
product_day = (
    recent[recent["StockCode"].isin(top_codes)]
    .groupby(["StockCode", "Date"], as_index=False)
    .agg(
        Daily_Units=("Quantity", "sum"),
        Transaction_Rows=("Invoice", "size"),
        Unique_Invoices=("Invoice", "nunique")
    )
)

# Add calendar days with zero recorded sales for every selected product.
all_dates = pd.date_range(start_date, end_date, freq="D")
product_index = pd.MultiIndex.from_product(
    [top_codes, all_dates],
    names=["StockCode", "Date"]
)

product_day = (
    product_day.set_index(["StockCode", "Date"])
    .reindex(product_index)
    .reset_index()
)

for col in ["Daily_Units", "Transaction_Rows", "Unique_Invoices"]:
    product_day[col] = product_day[col].fillna(0)

# Summarize demand variability, including zero-sales calendar days.
variability = (
    product_day.groupby("StockCode")
    .agg(
        Calendar_Days=("Date", "size"),
        Average_Daily_Units=("Daily_Units", "mean"),
        Median_Daily_Units=("Daily_Units", "median"),
        Daily_Units_Std=("Daily_Units", "std"),
        P95_Daily_Units=("Daily_Units", lambda s: s.quantile(0.95)),
        Maximum_Daily_Units=("Daily_Units", "max"),
        Days_With_Sales=("Daily_Units", lambda s: (s > 0).sum()),
        Total_Units=("Daily_Units", "sum")
    )
    .reset_index()
)

variability["Max_To_Mean_Ratio"] = (
    variability["Maximum_Daily_Units"]
    / variability["Average_Daily_Units"].replace(0, np.nan)
)

variability = variability.merge(
    top10[["StockCode", "Description", "Forecast_30Day_Demand_Units",
           "Safety_Stock_Units", "Estimated_Reorder_Point_Units"]],
    on="StockCode",
    how="left",
    validate="one_to_one"
)

variability = variability.sort_values(
    "Forecast_30Day_Demand_Units",
    ascending=False
)

# Show the highest daily sales for StockCode 23843.
focus_code = "23843"
focus_daily = (
    product_day[product_day["StockCode"].astype(str) == focus_code]
    .sort_values("Daily_Units", ascending=False)
    .head(10)
)

# Inspect the underlying invoice lines on that product's top sales dates.
focus_dates = focus_daily.loc[focus_daily["Daily_Units"] > 0, "Date"]

focus_lines = recent[
    (recent["StockCode"].astype(str) == focus_code)
    & (recent["Date"].isin(focus_dates))
].copy()

focus_lines = focus_lines.sort_values(
    ["Date", "Quantity"],
    ascending=[False, False]
)

# Save the diagnostics separately.
variability_path = REPORT_DIR / "product_demand_variability_review.csv"
focus_daily_path = REPORT_DIR / "product_23843_top_sales_days.csv"
focus_lines_path = REPORT_DIR / "product_23843_top_day_invoice_lines.csv"

variability.to_csv(variability_path, index=False)
focus_daily.to_csv(focus_daily_path, index=False)
focus_lines.to_csv(focus_lines_path, index=False)

print("=" * 70)
print("RETAILPULSE - PRODUCT DEMAND VARIABILITY REVIEW")
print("=" * 70)
print(f"Review window: {start_date.date()} to {end_date.date()}")

print("\nTop 10 products: mean, median and peak daily demand")
display(
    variability[[
        "StockCode", "Description", "Total_Units",
        "Average_Daily_Units", "Median_Daily_Units",
        "Daily_Units_Std", "P95_Daily_Units",
        "Maximum_Daily_Units", "Days_With_Sales",
        "Max_To_Mean_Ratio", "Safety_Stock_Units"
    ]].round(2)
)

print("\nHighest sales days for PAPER CRAFT LITTLE BIRDIE (23843):")
display(focus_daily.round(2))

print("\nUnderlying invoice lines on those high-sales days:")
display(
    focus_lines[[
        "Date", "Invoice", "StockCode", "Description",
        "Quantity", "Price", "Revenue", "Country"
    ]].head(30)
)

print("\nSaved diagnostic files:")
print(variability_path)
print(focus_daily_path)
print(focus_lines_path)
print("\nNo source or master data was modified.")



In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "inventory"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "retail_sales_final.csv"
TARGET_INVOICE = "581483"

columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country", "Revenue"
]

df = pd.read_csv(
    INPUT_PATH,
    usecols=columns,
    dtype={"Invoice": "string", "StockCode": "string"},
    low_memory=False
)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Revenue_Calculated"] = df["Quantity"] * df["Price"]

invoice = df[
    df["Invoice"].str.strip().eq(TARGET_INVOICE)
].copy()

if invoice.empty:
    raise ValueError(
        f"Invoice {TARGET_INVOICE} was not found. "
        "Check the invoice ID and source file."
    )

# Summary of the full invoice
invoice_summary = pd.DataFrame([{
    "Invoice": TARGET_INVOICE,
    "Transaction_Rows": len(invoice),
    "Unique_Products": invoice["StockCode"].nunique(),
    "Invoice_Date": (
        invoice["InvoiceDate"].min().date()
        if invoice["InvoiceDate"].notna().any() else None
    ),
    "Customer_IDs": invoice["Customer ID"].nunique(),
    "Countries": ", ".join(
        invoice["Country"].dropna().astype(str).unique()
    ),
    "Total_Recorded_Quantity": invoice["Quantity"].sum(),
    "Total_Calculated_Revenue": invoice["Revenue_Calculated"].sum(),
    "Target_Product_Quantity": invoice.loc[
        invoice["StockCode"].eq("23843"), "Quantity"
    ].sum()
}])

# Display all lines with largest quantities first
invoice_lines = invoice.sort_values(
    "Quantity", ascending=False
)[[
    "Invoice", "InvoiceDate", "Customer ID", "Country",
    "StockCode", "Description", "Quantity", "Price",
    "Revenue", "Revenue_Calculated"
]]

summary_path = REPORT_DIR / "invoice_581483_summary.csv"
lines_path = REPORT_DIR / "invoice_581483_lines.csv"

invoice_summary.to_csv(summary_path, index=False)
invoice_lines.to_csv(lines_path, index=False)

print("=" * 65)
print("RETAILPULSE - BULK TRANSACTION INVESTIGATION")
print("=" * 65)

print("\nInvoice summary:")
display(invoice_summary)

print("\nAll lines on invoice 581483:")
display(invoice_lines)

print("\nSaved review files:")
print(summary_path)
print(lines_path)

print("\nThe master dataset was not modified.")



In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "inventory"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = PROCESSED_DIR / "retail_sales_final.csv"
BASELINE_PATH = REPORT_DIR / "product_demand_reorder_baseline.csv"

INVOICE_TO_REVIEW = "581483"
TARGET_STOCK_CODE = "23843"
HISTORY_DAYS = 90
LEAD_TIME_DAYS = 14
REVIEW_PERIOD_DAYS = 7
Z_FACTOR = 1.65

columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country", "Revenue"
]

df = pd.read_csv(
    MASTER_PATH,
    usecols=columns,
    dtype={"Invoice": "string", "StockCode": "string"},
    low_memory=False
)

baseline = pd.read_csv(
    BASELINE_PATH,
    dtype={"StockCode": "string"}
)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Date"] = df["InvoiceDate"].dt.normalize()

eligible = df[
    df["Date"].notna()
    & df["StockCode"].notna()
    & (df["Quantity"] > 0)
    & (df["Price"] > 0)
].copy()

eligible["Revenue_Calculated"] = eligible["Quantity"] * eligible["Price"]

end_date = eligible["Date"].max()
start_date = end_date - pd.Timedelta(days=HISTORY_DAYS - 1)
calendar = pd.date_range(start_date, end_date, freq="D")

recent = eligible[eligible["Date"].between(start_date, end_date)].copy()

# Compare the original scenario with a clearly labelled sensitivity scenario.
scenarios = {
    "As recorded": recent.copy(),
    "Exclude invoice 581483 for sensitivity only": recent[
        ~recent["Invoice"].astype("string").str.strip().eq(INVOICE_TO_REVIEW)
    ].copy()
}

# Inspect the existing top 10 plus the product under investigation.
selected_codes = set(baseline.nlargest(
    10, "Forecast_30Day_Demand_Units"
)["StockCode"].astype(str))
selected_codes.add(TARGET_STOCK_CODE)

description_lookup = (
    recent.sort_values("Date")
    .drop_duplicates("StockCode", keep="last")
    .set_index("StockCode")["Description"]
    .to_dict()
)

comparison_rows = []
scenario_summary_rows = []

for scenario_name, scenario_data in scenarios.items():
    # Aggregate each product's daily units. Reindex to all calendar days so
    # days without recorded sales count as zero in both scenarios.
    units = scenario_data.pivot_table(
        index="Date",
        columns="StockCode",
        values="Quantity",
        aggfunc="sum",
        fill_value=0
    )

    units = units.reindex(
        index=calendar,
        columns=sorted(selected_codes),
        fill_value=0
    ).fillna(0)

    mean_units = units.mean(axis=0)
    std_units = units.std(axis=0, ddof=1).fillna(0)

    for code in sorted(selected_codes):
        average_daily = float(mean_units.get(code, 0))
        daily_std = float(std_units.get(code, 0))

        safety_stock = int(np.ceil(
            Z_FACTOR * daily_std * np.sqrt(LEAD_TIME_DAYS)
        ))
        reorder_point = int(np.ceil(
            average_daily * LEAD_TIME_DAYS + safety_stock
        ))
        target_stock = int(np.ceil(
            average_daily * (LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS)
            + Z_FACTOR * daily_std
            * np.sqrt(LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS)
        ))

        if code == TARGET_STOCK_CODE and average_daily == 0:
            interpretation = (
                "No eligible sales remain in this scenario; "
                "manual review required, not a zero-stock recommendation."
            )
        elif code == TARGET_STOCK_CODE:
            interpretation = (
                "Recorded demand is dominated by invoice 581483; "
                "verify the transaction before relying on the estimate."
            )
        else:
            interpretation = "Sensitivity estimate based on eligible sales."

        comparison_rows.append({
            "Scenario": scenario_name,
            "StockCode": code,
            "Description": description_lookup.get(code, "UNKNOWN"),
            "Average_Daily_Units": average_daily,
            "Forecast_30Day_Demand_Units": average_daily * 30,
            "Daily_Units_Std": daily_std,
            "Safety_Stock_Units": safety_stock,
            "Estimated_Reorder_Point_Units": reorder_point,
            "Estimated_Target_Stock_Units": target_stock,
            "Interpretation": interpretation
        })

    scenario_summary_rows.append({
        "Scenario": scenario_name,
        "Window_Start": start_date.date(),
        "Window_End": end_date.date(),
        "Eligible_Transaction_Rows": len(scenario_data),
        "Eligible_Units": scenario_data["Quantity"].sum(),
        "Eligible_Revenue": scenario_data["Revenue_Calculated"].sum(),
        "Target_Product_Units": scenario_data.loc[
            scenario_data["StockCode"].eq(TARGET_STOCK_CODE),
            "Quantity"
        ].sum()
    })

comparison = pd.DataFrame(comparison_rows)
scenario_summary = pd.DataFrame(scenario_summary_rows)

comparison_path = REPORT_DIR / "inventory_outlier_sensitivity_comparison.csv"
summary_path = REPORT_DIR / "inventory_outlier_sensitivity_summary.csv"

comparison.to_csv(comparison_path, index=False)
scenario_summary.to_csv(summary_path, index=False)

print("=" * 70)
print("RETAILPULSE - INVENTORY OUTLIER SENSITIVITY ANALYSIS")
print("=" * 70)

print("\nScenario summary:")
display(scenario_summary.round(2))

print("\nProduct 23843 under both scenarios:")
display(
    comparison[comparison["StockCode"] == TARGET_STOCK_CODE].round(2)
)

print("\nOther products in the original top 10:")
display(
    comparison[
        (comparison["StockCode"] != TARGET_STOCK_CODE)
        & comparison["Scenario"].eq("As recorded")
    ].sort_values("Forecast_30Day_Demand_Units", ascending=False).round(2)
)

print("\nSaved reports:")
print(comparison_path)
print(summary_path)
print("\nThe master dataset was not modified.")




In [ ]:

from pathlib import Path
import gc
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
RAW_PATH = PROJECT_DIR / "data" / "raw" / "online_retail_II.xlsx"
MASTER_PATH = PROJECT_DIR / "data" / "processed" / "retail_sales_final.csv"

REPORT_DIR = PROJECT_DIR / "reports" / "inventory"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Original workbook not found: {RAW_PATH}")

if not MASTER_PATH.exists():
    raise FileNotFoundError(f"Current master dataset not found: {MASTER_PATH}")

# Planning assumptions only; actual supplier lead times are not available.
HISTORY_DAYS = 90
LEAD_TIME_DAYS = 14
REVIEW_PERIOD_DAYS = 7
Z_FACTOR = 1.65

SHEETS = ["Year 2009-2010", "Year 2010-2011"]

SOURCE_COLUMNS = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country"
]

# Use the current master only to determine the final date to investigate.
# Product identifiers and transactions will come from the original workbook.
master_dates = pd.read_csv(
    MASTER_PATH,
    usecols=["InvoiceDate"]
)["InvoiceDate"]

master_dates = pd.to_datetime(master_dates, errors="coerce")
end_date = master_dates.max().normalize()

if pd.isna(end_date):
    raise ValueError("Could not determine the final date from the master data.")

start_date = end_date - pd.Timedelta(days=HISTORY_DAYS - 1)
calendar = pd.date_range(start_date, end_date, freq="D")

# --------------------------------------------------
# 1. Read one source sheet at a time.
# Preserve original StockCodes: only trim whitespace and standardize case.
# Do not remove suffix letters or strip other code characters.
# --------------------------------------------------
recent_parts = []

for sheet_name in SHEETS:
    print(f"Reading source sheet: {sheet_name}")

    source_sheet = pd.read_excel(
        RAW_PATH,
        sheet_name=sheet_name,
        usecols=SOURCE_COLUMNS,
        dtype={"Invoice": "string", "StockCode": "string"}
    )

    source_sheet["InvoiceDate"] = pd.to_datetime(
        source_sheet["InvoiceDate"], errors="coerce"
    )
    source_sheet["Quantity"] = pd.to_numeric(
        source_sheet["Quantity"], errors="coerce"
    )
    source_sheet["Price"] = pd.to_numeric(
        source_sheet["Price"], errors="coerce"
    )

    source_sheet["StockCode"] = (
        source_sheet["StockCode"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    source_sheet["Date"] = source_sheet["InvoiceDate"].dt.normalize()

    # Keep only the recent window and eligible positive sales.
    source_sheet = source_sheet[
        source_sheet["Date"].between(start_date, end_date)
        & source_sheet["StockCode"].notna()
        & source_sheet["StockCode"].ne("")
        & (source_sheet["Quantity"] > 0)
        & (source_sheet["Price"] > 0)
    ].copy()

    if not source_sheet.empty:
        source_sheet["Sales_Revenue"] = (
            source_sheet["Quantity"] * source_sheet["Price"]
        )
        recent_parts.append(source_sheet)

    del source_sheet
    gc.collect()

if not recent_parts:
    raise ValueError("No eligible sales found in the original workbook window.")

recent = pd.concat(recent_parts, ignore_index=True)
del recent_parts
gc.collect()

# --------------------------------------------------
# 2. Build daily unit demand using original StockCodes
# Missing product/date combinations are represented as zero sales.
# --------------------------------------------------
daily_units = recent.pivot_table(
    index="Date",
    columns="StockCode",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

daily_units = daily_units.reindex(
    index=calendar,
    fill_value=0
).fillna(0)

mean_units = daily_units.mean(axis=0)
std_units = daily_units.std(axis=0, ddof=1).fillna(0)
max_daily_units = daily_units.max(axis=0)
days_with_sales = (daily_units > 0).sum(axis=0)

# --------------------------------------------------
# 3. Select representative descriptions and audit variations
# --------------------------------------------------
def most_common_description(values):
    clean = values.dropna().astype(str).str.strip()
    clean = clean[clean.ne("")]
    modes = clean.mode()
    return modes.iloc[0] if not modes.empty else "UNKNOWN"

description_by_code = (
    recent.groupby("StockCode")["Description"]
    .agg(most_common_description)
)

description_variant_count = (
    recent.groupby("StockCode")["Description"]
    .nunique(dropna=True)
)

products = pd.DataFrame({
    "StockCode": mean_units.index.astype(str),
    "Average_Daily_Units": mean_units.values,
    "Daily_Units_Std": std_units.values,
    "Maximum_Daily_Units": max_daily_units.values,
    "Days_With_Sales": days_with_sales.values
})

products["Description"] = (
    products["StockCode"].map(description_by_code).fillna("UNKNOWN")
)
products["Description_Variant_Count"] = (
    products["StockCode"].map(description_variant_count).fillna(0).astype(int)
)

# Record the known bulk invoice for review, but do not remove it.
bulk_invoice = recent[
    recent["Invoice"].astype("string").str.strip().eq("581483")
].copy()

bulk_units_by_code = (
    bulk_invoice.groupby("StockCode")["Quantity"].sum()
    if not bulk_invoice.empty
    else pd.Series(dtype=float)
)

products["Invoice_581483_Units"] = (
    products["StockCode"].map(bulk_units_by_code).fillna(0)
)

# --------------------------------------------------
# 4. Calculate provisional planning estimates
# --------------------------------------------------
products["Forecast_30Day_Demand_Units"] = (
    products["Average_Daily_Units"] * 30
)

products["Average_Daily_Revenue"] = (
    recent.groupby("StockCode")["Sales_Revenue"].sum()
    .reindex(products["StockCode"])
    .to_numpy()
    / HISTORY_DAYS
)

products["Forecast_30Day_Revenue"] = (
    products["Average_Daily_Revenue"] * 30
)

products["Assumed_Lead_Time_Days"] = LEAD_TIME_DAYS
products["Assumed_Review_Period_Days"] = REVIEW_PERIOD_DAYS
products["Safety_Factor_Z"] = Z_FACTOR

products["Safety_Stock_Units"] = np.ceil(
    Z_FACTOR * products["Daily_Units_Std"] * np.sqrt(LEAD_TIME_DAYS)
)

products["Estimated_Reorder_Point_Units"] = np.ceil(
    products["Average_Daily_Units"] * LEAD_TIME_DAYS
    + products["Safety_Stock_Units"]
)

coverage_days = LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS

products["Estimated_Target_Stock_Units"] = np.ceil(
    products["Average_Daily_Units"] * coverage_days
    + Z_FACTOR * products["Daily_Units_Std"] * np.sqrt(coverage_days)
)

# An actual reorder quantity cannot be calculated without stock on hand.
products["Current_Stock_On_Hand"] = pd.NA
products["Actual_Supplier_Lead_Time_Days"] = pd.NA
products["Suggested_Reorder_Quantity"] = pd.NA

# Flag suspicious concentration in one day and known invoice 581483.
products["Max_To_Mean_Daily_Units_Ratio"] = (
    products["Maximum_Daily_Units"]
    / products["Average_Daily_Units"].replace(0, np.nan)
)

products["Review_Status"] = "PROVISIONAL: stock-on-hand and actual lead time required"

bulk_mask = products["Invoice_581483_Units"] > 0
products.loc[bulk_mask, "Review_Status"] = (
    "MANUAL REVIEW: contains invoice 581483; verify bulk quantity"
)

spike_mask = (
    products["Max_To_Mean_Daily_Units_Ratio"] > 5
) & ~bulk_mask

products.loc[spike_mask, "Review_Status"] = (
    "REVIEW: daily demand is highly concentrated; verify transactions"
)

products["Operational_Recommendation_Ready"] = False

products["Window_Start"] = start_date.date().isoformat()
products["Window_End"] = end_date.date().isoformat()
products["Calendar_Days_Analysed"] = HISTORY_DAYS

products = products.sort_values(
    "Forecast_30Day_Demand_Units",
    ascending=False
).reset_index(drop=True)

# --------------------------------------------------
# 5. Save a separate audit of StockCodes with multiple descriptions
# --------------------------------------------------
description_audit = (
    recent.dropna(subset=["Description"])
    .assign(Description=lambda d: d["Description"].astype(str).str.strip())
    .groupby(["StockCode", "Description"], as_index=False)
    .size()
    .sort_values(["StockCode", "size"], ascending=[True, False])
)

variant_codes = set(
    products.loc[
        products["Description_Variant_Count"] > 1, "StockCode"
    ]
)

description_audit = description_audit[
    description_audit["StockCode"].isin(variant_codes)
].copy()

# --------------------------------------------------
# 6. Save new files without overwriting previous results
# --------------------------------------------------
baseline_path = (
    REPORT_DIR / "product_demand_reorder_baseline_original_stockcodes.csv"
)
audit_path = (
    REPORT_DIR / "stockcode_description_variant_audit.csv"
)
summary_path = (
    REPORT_DIR / "inventory_original_stockcode_summary.csv"
)

products.to_csv(baseline_path, index=False)
description_audit.to_csv(audit_path, index=False)

suffix_mask = products["StockCode"].str.contains(
    r"[A-Z]$", regex=True, na=False
)

summary = pd.DataFrame([
    {"Metric": "Source workbook", "Value": str(RAW_PATH)},
    {"Metric": "Historical window start", "Value": str(start_date.date())},
    {"Metric": "Historical window end", "Value": str(end_date.date())},
    {"Metric": "Calendar days analysed", "Value": HISTORY_DAYS},
    {"Metric": "Eligible source transaction rows", "Value": len(recent)},
    {"Metric": "Eligible units in window", "Value": round(recent["Quantity"].sum(), 2)},
    {"Metric": "Eligible revenue in window", "Value": round(recent["Sales_Revenue"].sum(), 2)},
    {"Metric": "Original StockCodes with eligible sales", "Value": len(products)},
    {"Metric": "StockCodes ending in a letter", "Value": int(suffix_mask.sum())},
    {"Metric": "StockCodes with multiple descriptions", "Value": len(variant_codes)},
    {"Metric": "Rows from invoice 581483 in the window", "Value": len(bulk_invoice)},
    {"Metric": "Units on invoice 581483 in the window", "Value": round(bulk_invoice["Quantity"].sum(), 2)}
])

summary.to_csv(summary_path, index=False)

print("=" * 70)
print("RETAILPULSE - ORIGINAL STOCKCODE INVENTORY BASELINE")
print("=" * 70)
display(summary)

print("\nTop 10 products by estimated 30-day unit demand:")
display(
    products[[
        "StockCode", "Description", "Description_Variant_Count",
        "Average_Daily_Units", "Forecast_30Day_Demand_Units",
        "Safety_Stock_Units", "Estimated_Reorder_Point_Units",
        "Review_Status"
    ]].head(10).round(2)
)

print("\nRows flagged for invoice 581483:")
display(products[products["Invoice_581483_Units"] > 0][[
    "StockCode", "Description", "Invoice_581483_Units",
    "Average_Daily_Units", "Forecast_30Day_Demand_Units",
    "Safety_Stock_Units", "Review_Status"
]].round(2))

print("\nSaved files:")
print(baseline_path)
print(audit_path)
print(summary_path)
print("\nThe master dataset and original workbook were not modified.")


In [ ]:

from pathlib import Path
import gc
import numpy as np
import pandas as pd

try:
    from prophet import Prophet
except ImportError as exc:
    raise ImportError(
        "Prophet is not installed. Run %pip install prophet, "
        "restart the kernel, and run this cell again."
    ) from exc

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
RAW_PATH = PROJECT_DIR / "data" / "raw" / "online_retail_II.xlsx"
INVENTORY_PATH = (
    PROJECT_DIR / "reports" / "inventory"
    / "product_demand_reorder_baseline_original_stockcodes.csv"
)
REPORT_DIR = PROJECT_DIR / "reports" / "forecasting"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SHEETS = ["Year 2009-2010", "Year 2010-2011"]
TEST_DAYS = 30
MIN_NONZERO_TRAIN_DAYS = 5

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Original workbook not found: {RAW_PATH}")
if not INVENTORY_PATH.exists():
    raise FileNotFoundError(f"Original-StockCode inventory baseline not found: {INVENTORY_PATH}")

# Select the 10 highest-demand products from the source-faithful inventory baseline.
baseline = pd.read_csv(INVENTORY_PATH, dtype={"StockCode": "string"})
selected = baseline.nlargest(10, "Forecast_30Day_Demand_Units").copy()
selected_codes = set(
    selected["StockCode"].astype("string").str.strip().str.upper()
)

selected_descriptions = (
    selected.set_index("StockCode")["Description"].to_dict()
)

source_columns = [
    "StockCode", "Description", "Quantity", "InvoiceDate", "Price"
]
parts = []
valid_source_dates = []

# Read one sheet at a time, retaining the original product codes.
for sheet_name in SHEETS:
    print(f"Reading source sheet: {sheet_name}")

    chunk = pd.read_excel(
        RAW_PATH,
        sheet_name=sheet_name,
        usecols=source_columns,
        dtype={"StockCode": "string"}
    )

    chunk["StockCode"] = (
        chunk["StockCode"].astype("string").str.strip().str.upper()
    )
    chunk["InvoiceDate"] = pd.to_datetime(
        chunk["InvoiceDate"], errors="coerce"
    )
    chunk["Quantity"] = pd.to_numeric(chunk["Quantity"], errors="coerce")
    chunk["Price"] = pd.to_numeric(chunk["Price"], errors="coerce")

    valid_source_dates.extend(
        chunk["InvoiceDate"].dropna().dt.normalize().tolist()
        if not chunk["InvoiceDate"].dropna().empty
        else []
    )

    chunk = chunk[
        chunk["StockCode"].isin(selected_codes)
        & chunk["InvoiceDate"].notna()
        & (chunk["Quantity"] > 0)
        & (chunk["Price"] > 0)
    ].copy()

    if not chunk.empty:
        chunk["Date"] = chunk["InvoiceDate"].dt.normalize()
        parts.append(chunk[["StockCode", "Description", "Date", "Quantity"]])

    del chunk
    gc.collect()

if not parts or not valid_source_dates:
    raise ValueError("No usable source transactions or dates were found.")

transactions = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

# Match the latest 30 calendar days from the full source data.
dataset_start = pd.Timestamp(min(valid_source_dates)).normalize()
dataset_end = pd.Timestamp(max(valid_source_dates)).normalize()
test_start = dataset_end - pd.Timedelta(days=TEST_DAYS - 1)
train_end = test_start - pd.Timedelta(days=1)

calendar = pd.date_range(dataset_start, dataset_end, freq="D")
train_dates = calendar[calendar <= train_end]
test_dates = calendar[calendar >= test_start]

if len(test_dates) != TEST_DAYS:
    raise ValueError(f"Expected {TEST_DAYS} test dates, found {len(test_dates)}.")

print(f"Training period: {train_dates.min().date()} to {train_dates.max().date()}")
print(f"Testing period:  {test_dates.min().date()} to {test_dates.max().date()}")

prediction_rows = []
metric_rows = []
eligibility_rows = []

def calculate_metrics(actual, predicted, sku, description, model_name):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    abs_error = np.abs(actual - predicted)
    actual_total = actual.sum()
    denominator = np.abs(actual) + np.abs(predicted)

    nonzero = actual > 0
    mape = (
        np.mean(abs_error[nonzero] / actual[nonzero]) * 100
        if nonzero.any() else np.nan
    )

    smape_terms = np.divide(
        2 * abs_error,
        denominator,
        out=np.zeros_like(abs_error),
        where=denominator != 0
    )

    return {
        "StockCode": sku,
        "Description": description,
        "Model": model_name,
        "Test_Days": len(actual),
        "MAE_Units_Per_Day": float(np.mean(abs_error)),
        "RMSE_Units_Per_Day": float(np.sqrt(np.mean((actual - predicted) ** 2))),
        "WAPE_Percent": (
            float(abs_error.sum() / actual_total * 100)
            if actual_total > 0 else np.nan
        ),
        "MAPE_On_Positive_Actual_Days_Percent": float(mape) if pd.notna(mape) else np.nan,
        "SMAPE_Percent": float(np.mean(smape_terms) * 100),
        "Actual_Test_Units": float(actual_total),
        "Forecast_Test_Units": float(predicted.sum()),
        "Positive_Actual_Test_Days": int(nonzero.sum())
    }

# --------------------------------------------------
# Fit and evaluate product-level models
# --------------------------------------------------
for sku in sorted(selected_codes):
    product_tx = transactions[transactions["StockCode"].eq(sku)].copy()

    description = selected_descriptions.get(sku, "UNKNOWN")
    daily_actual = (
        product_tx.groupby("Date")["Quantity"].sum()
        .reindex(calendar, fill_value=0)
        .astype(float)
    )

    y_train = daily_actual.loc[train_dates]
    y_test = daily_actual.loc[test_dates]

    nonzero_training_days = int((y_train > 0).sum())
    unique_training_values = int(y_train.nunique())

    # Weekly seasonal-naive benchmark:
    # forecast each future day from the same weekday in the prior week,
    # recursively using predictions once a test day has been forecast.
    history = {date: float(value) for date, value in y_train.items()}
    naive_predictions = []

    for date in test_dates:
        lag_date = date - pd.Timedelta(days=7)
        value = history.get(lag_date, 0.0)
        value = max(0.0, float(value))
        naive_predictions.append(value)
        history[date] = value

    actual_values = y_test.to_numpy(dtype=float)
    naive_predictions = np.asarray(naive_predictions, dtype=float)

    metric_rows.append(
        calculate_metrics(
            actual_values, naive_predictions,
            sku, description, "Weekly Seasonal Naive"
        )
    )

    # A product with no variation or almost no historical sales cannot
    # support a reliable model from these data. Keep it flagged.
    can_fit_prophet = (
        nonzero_training_days >= MIN_NONZERO_TRAIN_DAYS
        and unique_training_values > 1
    )

    if can_fit_prophet:
        fit_df = pd.DataFrame({
            "ds": y_train.index,
            "y": y_train.to_numpy(dtype=float)
        })

        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            interval_width=0.95
        )
        model.fit(fit_df)

        future = pd.DataFrame({"ds": test_dates})
        forecast = model.predict(future)

        prophet_predictions = np.clip(
            forecast["yhat"].to_numpy(dtype=float), 0, None
        )

        metric_rows.append(
            calculate_metrics(
                actual_values, prophet_predictions,
                sku, description, "Prophet"
            )
        )

        status = "Prophet fitted"
    else:
        prophet_predictions = np.full(len(test_dates), np.nan)
        status = "Insufficient historical sales/variation; Prophet not fitted"

    eligibility_rows.append({
        "StockCode": sku,
        "Description": description,
        "Nonzero_Training_Days": nonzero_training_days,
        "Unique_Daily_Quantity_Values_In_Training": unique_training_values,
        "Actual_Test_Units": float(actual_values.sum()),
        "Status": status
    })

    for i, date in enumerate(test_dates):
        prediction_rows.append({
            "StockCode": sku,
            "Description": description,
            "Date": date,
            "Actual_Daily_Units": actual_values[i],
            "Weekly_Seasonal_Naive_Forecast": naive_predictions[i],
            "Prophet_Forecast": prophet_predictions[i],
            "Forecast_Status": status
        })

# --------------------------------------------------
# Save predictions, metrics and model eligibility
# --------------------------------------------------
predictions_df = pd.DataFrame(prediction_rows)
metrics_df = pd.DataFrame(metric_rows)
eligibility_df = pd.DataFrame(eligibility_rows)

predictions_path = REPORT_DIR / "top10_product_demand_predictions.csv"
metrics_path = REPORT_DIR / "top10_product_demand_metrics.csv"
eligibility_path = REPORT_DIR / "top10_product_model_eligibility.csv"

predictions_df.to_csv(predictions_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
eligibility_df.to_csv(eligibility_path, index=False)

print("\n" + "=" * 70)
print("RETAILPULSE - PRODUCT-LEVEL DEMAND FORECAST RESULTS")
print("=" * 70)

print("\nModel eligibility:")
display(eligibility_df)

print("\nForecast metrics for each product:")
display(
    metrics_df.sort_values(["StockCode", "Model"]).round(3)
)

print("\nPredictions for PAPER CRAFT LITTLE BIRDIE (23843):")
display(
    predictions_df[predictions_df["StockCode"].eq("23843")].tail(10)
)

print("\nSaved files:")
print(predictions_path)
print(metrics_path)
print(eligibility_path)
print("\nOriginal workbook and master dataset were not modified.")


In [ ]:

from pathlib import Path
import gc
import numpy as np
import pandas as pd

from prophet import Prophet

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
RAW_PATH = PROJECT_DIR / "data" / "raw" / "online_retail_II.xlsx"
BASELINE_PATH = (
    PROJECT_DIR / "reports" / "inventory"
    / "product_demand_reorder_baseline_original_stockcodes.csv"
)
REPORT_DIR = PROJECT_DIR / "reports" / "forecasting"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SHEETS = ["Year 2009-2010", "Year 2010-2011"]
TEST_DAYS = 30
VALIDATION_DAYS = 30
MIN_NONZERO_DAYS = 5

if not RAW_PATH.exists() or not BASELINE_PATH.exists():
    raise FileNotFoundError("Original workbook or inventory baseline is missing.")

baseline = pd.read_csv(BASELINE_PATH, dtype={"StockCode": "string"})
top_products = baseline.nlargest(10, "Forecast_30Day_Demand_Units").copy()
selected_codes = set(
    top_products["StockCode"].astype(str).str.strip().str.upper()
)
description_map = dict(zip(
    top_products["StockCode"].astype(str),
    top_products["Description"]
))

# Read original data, preserving the original StockCodes.
parts = []
all_min_dates = []
all_max_dates = []

for sheet_name in SHEETS:
    print(f"Reading source sheet: {sheet_name}")
    raw = pd.read_excel(
        RAW_PATH,
        sheet_name=sheet_name,
        usecols=["StockCode", "Quantity", "InvoiceDate", "Price"],
        dtype={"StockCode": "string"}
    )

    raw["StockCode"] = (
        raw["StockCode"].astype("string").str.strip().str.upper()
    )
    raw["InvoiceDate"] = pd.to_datetime(
        raw["InvoiceDate"], errors="coerce"
    )
    raw["Quantity"] = pd.to_numeric(raw["Quantity"], errors="coerce")
    raw["Price"] = pd.to_numeric(raw["Price"], errors="coerce")

    valid_dates = raw["InvoiceDate"].dropna().dt.normalize()
    if not valid_dates.empty:
        all_min_dates.append(valid_dates.min())
        all_max_dates.append(valid_dates.max())

    raw = raw[
        raw["StockCode"].isin(selected_codes)
        & raw["InvoiceDate"].notna()
        & (raw["Quantity"] > 0)
        & (raw["Price"] > 0)
    ].copy()

    if not raw.empty:
        raw["Date"] = raw["InvoiceDate"].dt.normalize()
        parts.append(raw[["StockCode", "Date", "Quantity"]])

    del raw
    gc.collect()

if not parts or not all_min_dates or not all_max_dates:
    raise ValueError("No eligible product history or dates were found.")

transactions = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

start_date = min(all_min_dates)
end_date = max(all_max_dates)
calendar = pd.date_range(start_date, end_date, freq="D")

test_start = end_date - pd.Timedelta(days=TEST_DAYS - 1)
train_end = test_start - pd.Timedelta(days=1)
train_dates = calendar[calendar <= train_end]

if len(train_dates) <= VALIDATION_DAYS:
    raise ValueError("Not enough training dates for the validation split.")

validation_dates = train_dates[-VALIDATION_DAYS:]
model_fit_end = validation_dates[0] - pd.Timedelta(days=1)
test_dates = calendar[calendar >= test_start]

print(f"Model-fitting period ends: {model_fit_end.date()}")
print(
    f"Validation period: {validation_dates.min().date()} "
    f"to {validation_dates.max().date()}"
)
print(f"Final test period: {test_dates.min().date()} to {test_dates.max().date()}")

def weekly_seasonal_forecast(history_series, future_dates):
    history = {
        date: float(value)
        for date, value in history_series.items()
    }
    preds = []
    for date in future_dates:
        lag_date = date - pd.Timedelta(days=7)
        prediction = max(0.0, history.get(lag_date, 0.0))
        preds.append(prediction)
        history[date] = prediction
    return np.asarray(preds, dtype=float)

def wape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    total_actual = actual.sum()
    if total_actual <= 0:
        return np.nan
    return float(np.abs(actual - predicted).sum() / total_actual * 100)

def error_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        "MAE_Units_Per_Day": float(np.mean(np.abs(actual - predicted))),
        "RMSE_Units_Per_Day": float(
            np.sqrt(np.mean((actual - predicted) ** 2))
        ),
        "WAPE_Percent": wape(actual, predicted),
        "Actual_Test_Units": float(actual.sum()),
        "Forecast_Test_Units": float(predicted.sum())
    }

validation_rows = []
test_rows = []
prediction_rows = []

for sku in sorted(selected_codes):
    sku = str(sku)
    daily = (
        transactions[transactions["StockCode"].eq(sku)]
        .groupby("Date")["Quantity"].sum()
        .reindex(calendar, fill_value=0)
        .astype(float)
    )

    description = description_map.get(sku, "UNKNOWN")
    fit_history = daily.loc[daily.index <= model_fit_end]
    validation_actual = daily.loc[validation_dates].to_numpy(dtype=float)
    full_train = daily.loc[daily.index <= train_end]
    test_actual = daily.loc[test_dates].to_numpy(dtype=float)

    candidate_validation = []
    candidate_config = {}

    # Candidate 1: simple weekly seasonal baseline
    naive_validation = weekly_seasonal_forecast(
        fit_history, validation_dates
    )
    candidate_validation.append((
        "Weekly Seasonal Naive",
        wape(validation_actual, naive_validation),
        None
    ))
    candidate_config["Weekly Seasonal Naive"] = None

    # Candidates 2 and 3: simpler Prophet seasonality configurations
    nonzero_days = int((fit_history > 0).sum())
    can_fit = nonzero_days >= MIN_NONZERO_DAYS and fit_history.nunique() > 1

    if can_fit:
        fit_df = pd.DataFrame({
            "ds": fit_history.index,
            "y": fit_history.to_numpy(dtype=float)
        })

        for model_name, yearly_setting in [
            ("Prophet - No Yearly Seasonality", False),
            ("Prophet - Low-Order Yearly Seasonality", 3)
        ]:
            model = Prophet(
                yearly_seasonality=yearly_setting,
                weekly_seasonality=True,
                daily_seasonality=False
            )
            model.fit(fit_df)

            future = pd.DataFrame({"ds": validation_dates})
            forecast = model.predict(future)
            valid_prediction = np.clip(
                forecast["yhat"].to_numpy(dtype=float), 0, None
            )

            score = wape(validation_actual, valid_prediction)
            candidate_validation.append((model_name, score, yearly_setting))
            candidate_config[model_name] = yearly_setting

            del model
            gc.collect()

    # Model selection is based only on the validation period.
    valid_candidates = [
        item for item in candidate_validation
        if np.isfinite(item[1])
    ]

    if not valid_candidates:
        chosen_name = "Weekly Seasonal Naive"
        chosen_setting = None
        chosen_validation_wape = np.nan
    else:
        chosen_name, chosen_validation_wape, chosen_setting = min(
            valid_candidates, key=lambda item: item[1]
        )

    for model_name, score, _ in candidate_validation:
        validation_rows.append({
            "StockCode": sku,
            "Description": description,
            "Model": model_name,
            "Validation_WAPE_Percent": score,
            "Selected_By_Validation": model_name == chosen_name,
            "Nonzero_Model_Fitting_Days": nonzero_days
        })

    # Forecast the untouched final test period with the selected method.
    if chosen_name == "Weekly Seasonal Naive":
        test_prediction = weekly_seasonal_forecast(full_train, test_dates)
    else:
        full_fit_df = pd.DataFrame({
            "ds": full_train.index,
            "y": full_train.to_numpy(dtype=float)
        })

        final_model = Prophet(
            yearly_seasonality=chosen_setting,
            weekly_seasonality=True,
            daily_seasonality=False
        )
        final_model.fit(full_fit_df)

        test_forecast = final_model.predict(
            pd.DataFrame({"ds": test_dates})
        )
        test_prediction = np.clip(
            test_forecast["yhat"].to_numpy(dtype=float), 0, None
        )

        del final_model
        gc.collect()

    test_result = error_metrics(test_actual, test_prediction)
    test_rows.append({
        "StockCode": sku,
        "Description": description,
        "Selected_Model": chosen_name,
        "Validation_WAPE_Percent": chosen_validation_wape,
        "Nonzero_Fitting_Days": nonzero_days,
        **test_result
    })

    for i, date in enumerate(test_dates):
        prediction_rows.append({
            "StockCode": sku,
            "Description": description,
            "Date": date,
            "Actual_Daily_Units": test_actual[i],
            "Selected_Model_Forecast": test_prediction[i],
            "Selected_Model": chosen_name
        })

validation_results = pd.DataFrame(validation_rows)
test_results = pd.DataFrame(test_rows)
test_predictions = pd.DataFrame(prediction_rows)

validation_path = REPORT_DIR / "top10_product_validation_model_comparison_v2.csv"
test_metrics_path = REPORT_DIR / "top10_product_test_metrics_v2.csv"
test_predictions_path = REPORT_DIR / "top10_product_test_predictions_v2.csv"

validation_results.to_csv(validation_path, index=False)
test_results.to_csv(test_metrics_path, index=False)
test_predictions.to_csv(test_predictions_path, index=False)

print("\n" + "=" * 70)
print("RETAILPULSE - VALIDATION-SELECTED PRODUCT FORECASTS")
print("=" * 70)

print("\nValidation comparison:")
display(
    validation_results.sort_values(
        ["StockCode", "Validation_WAPE_Percent"]
    ).round(3)
)

print("\nFinal test metrics for models selected using validation only:")
display(test_results.round(3))

print("\nSaved files:")
print(validation_path)
print(test_metrics_path)
print(test_predictions_path)
print("\nThe source workbook and master dataset were not modified.")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
REPORT_DIR = PROJECT_DIR / "reports" / "forecasting"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = REPORT_DIR / "ensemble_30day_predictions.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Forecast comparison file not found: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH, parse_dates=["Date"])

required = [
    "Date", "Daily_Revenue", "Prophet_Forecast",
    "LSTM_Forecast", "Ensemble_Forecast"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required columns are missing: {missing}")

df = df.sort_values("Date").reset_index(drop=True)

if len(df) != 30:
    raise ValueError(f"Expected 30 test days; found {len(df)}.")

# Invoice 581483: 80,995 units × 2.08 = 168,469.60.
# This adjustment is for sensitivity analysis only.
# It does NOT change the recorded dataset or official actuals.
invoice_date = pd.Timestamp("2011-12-09")
invoice_revenue = 168469.60

sensitivity = df.copy()
target_rows = sensitivity["Date"].dt.normalize().eq(invoice_date)

if target_rows.sum() != 1:
    raise ValueError(
        "Expected exactly one test-period row for 2011-12-09."
    )

if sensitivity.loc[target_rows, "Daily_Revenue"].iloc[0] < invoice_revenue:
    raise ValueError(
        "Invoice revenue exceeds total recorded revenue for that day. "
        "Check the source before calculating the sensitivity scenario."
    )

sensitivity["Revenue_Actual_Sensitivity_Only"] = sensitivity["Daily_Revenue"]
sensitivity.loc[target_rows, "Revenue_Actual_Sensitivity_Only"] -= invoice_revenue

def score(actual, forecast, model_name, scenario):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    errors = np.abs(actual - forecast)
    total_actual = actual.sum()
    denom = np.abs(actual) + np.abs(forecast)

    positive_actual = actual > 0
    mape_positive = (
        np.mean(errors[positive_actual] / actual[positive_actual]) * 100
        if positive_actual.any() else np.nan
    )

    smape_terms = np.divide(
        2 * errors,
        denom,
        out=np.zeros_like(errors),
        where=denom != 0
    )

    return {
        "Scenario": scenario,
        "Model": model_name,
        "MAE": errors.mean(),
        "RMSE": np.sqrt(np.mean((actual - forecast) ** 2)),
        "WAPE_Percent": (
            errors.sum() / total_actual * 100
            if total_actual > 0 else np.nan
        ),
        "MAPE_Positive_Actual_Days_Percent": mape_positive,
        "SMAPE_Percent": smape_terms.mean() * 100,
        "Actual_Total_Revenue": total_actual,
        "Forecast_Total_Revenue": forecast.sum()
    }

prediction_columns = {
    "Prophet": "Prophet_Forecast",
    "LSTM": "LSTM_Forecast",
    "Equal-Weight Ensemble": "Ensemble_Forecast"
}

rows = []
for model_name, col in prediction_columns.items():
    rows.append(
        score(
            df["Daily_Revenue"],
            df[col],
            model_name,
            "As recorded"
        )
    )
    rows.append(
        score(
            sensitivity["Revenue_Actual_Sensitivity_Only"],
            df[col],
            model_name,
            "Invoice 581483 set aside for sensitivity only"
        )
    )

results = pd.DataFrame(rows)

output_path = REPORT_DIR / "forecast_invoice_sensitivity_metrics.csv"
results.to_csv(output_path, index=False)

print("=" * 70)
print("RETAILPULSE - FORECAST OUTLIER SENSITIVITY CHECK")
print("=" * 70)
print(f"Test period: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Invoice revenue considered in sensitivity: {invoice_revenue:,.2f}")
print("The model forecasts have not been changed.")

display(results.round(3))

print("\nSaved report:")
print(output_path)


In [ ]:


from pathlib import Path
import gc
import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "forecasting"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FULL_PATH = PROCESSED_DIR / "daily_sales_timeseries.csv"
TRAIN_PATH = PROCESSED_DIR / "forecast_train_30day.csv"
TEST_PATH = PROCESSED_DIR / "forecast_test_30day.csv"

for p in [FULL_PATH, TRAIN_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

daily = pd.read_csv(FULL_PATH, parse_dates=["Date"]).sort_values("Date")
train = pd.read_csv(TRAIN_PATH, parse_dates=["Date"]).sort_values("Date")
test = pd.read_csv(TEST_PATH, parse_dates=["Date"]).sort_values("Date")

series = daily.set_index("Date")["Daily_Revenue"].astype(float)

# The final 30 days of the training period are used for validation.
validation = train.tail(30).copy()
fit_period = train.iloc[:-30].copy()

validation_dates = validation["Date"].tolist()
test_dates = test["Date"].tolist()
fit_end = fit_period["Date"].max()
train_end = train["Date"].max()

if len(validation) != 30 or len(test) != 30:
    raise ValueError("Expected 30 validation days and 30 final test days.")

if fit_end + pd.Timedelta(days=1) != validation_dates[0]:
    raise ValueError("Validation split dates are not consecutive.")

print(f"Model-fitting period ends: {fit_end.date()}")
print(f"Validation: {validation_dates[0].date()} to {validation_dates[-1].date()}")
print(f"Final test: {test_dates[0].date()} to {test_dates[-1].date()}")

# --------------------------------------------------
# Feature creation: uses only values from dates before
# the date being predicted.
# --------------------------------------------------
LAGS = [1, 2, 3, 7, 14, 21, 28, 35, 56]
ROLLING_WINDOWS = [7, 14, 28, 56]

def is_black_friday(date):
    november_first = pd.Timestamp(year=date.year, month=11, day=1)
    first_thursday = november_first + pd.Timedelta(
        days=(3 - november_first.weekday()) % 7
    )
    thanksgiving = first_thursday + pd.Timedelta(days=21)
    black_friday = thanksgiving + pd.Timedelta(days=1)
    return int(date.normalize() == black_friday.normalize())

def make_features(date, history):
    date = pd.Timestamp(date).normalize()
    features = {}

    for lag in LAGS:
        features[f"lag_{lag}"] = float(
            history.get(date - pd.Timedelta(days=lag), 0.0)
        )

    for window in ROLLING_WINDOWS:
        prior_values = np.array([
            history.get(date - pd.Timedelta(days=i), 0.0)
            for i in range(1, window + 1)
        ], dtype=float)

        features[f"rolling_mean_{window}"] = float(prior_values.mean())
        features[f"rolling_std_{window}"] = float(prior_values.std())

    features["day_of_week"] = date.dayofweek
    features["day_of_month"] = date.day
    features["month"] = date.month
    features["is_weekend"] = int(date.dayofweek >= 5)
    features["is_month_start"] = int(date.is_month_start)
    features["is_month_end"] = int(date.is_month_end)
    features["is_december"] = int(date.month == 12)
    features["is_november"] = int(date.month == 11)
    features["is_black_friday"] = is_black_friday(date)
    features["is_christmas_day"] = int(date.month == 12 and date.day == 25)
    features["is_boxing_day"] = int(date.month == 12 and date.day == 26)

    year_length = 366 if date.is_leap_year else 365
    features["year_sin"] = np.sin(2 * np.pi * date.dayofyear / year_length)
    features["year_cos"] = np.cos(2 * np.pi * date.dayofyear / year_length)

    return features

def build_training_data(full_series, cutoff):
    available = full_series.loc[:cutoff]
    history = {pd.Timestamp(d): float(v) for d, v in available.items()}

    # Exclude the first 56 days because complete lag history is unavailable.
    target_dates = available.index[56:]
    X_rows = [make_features(d, history) for d in target_dates]
    X = pd.DataFrame(X_rows)
    y = available.loc[target_dates].to_numpy(dtype=float)

    return X, y

def recursive_ml_forecast(model, history_series, future_dates):
    history = {
        pd.Timestamp(d): float(v)
        for d, v in history_series.items()
    }
    predictions = []

    for date in future_dates:
        row = pd.DataFrame([make_features(date, history)])
        log_prediction = float(model.predict(row)[0])
        prediction = max(0.0, float(np.expm1(log_prediction)))
        predictions.append(prediction)
        history[pd.Timestamp(date)] = prediction

    return np.asarray(predictions, dtype=float)

def weekly_seasonal_forecast(history_series, future_dates):
    history = {
        pd.Timestamp(d): float(v)
        for d, v in history_series.items()
    }
    predictions = []

    for date in future_dates:
        date = pd.Timestamp(date)
        prediction = max(
            0.0,
            history.get(date - pd.Timedelta(days=7), 0.0)
        )
        predictions.append(prediction)
        history[date] = prediction

    return np.asarray(predictions, dtype=float)

def prophet_forecast(history_series, future_dates):
    prophet_train = pd.DataFrame({
        "ds": history_series.index,
        "y": history_series.to_numpy(dtype=float)
    })

    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=True,
        daily_seasonality=False
    )
    model.fit(prophet_train)

    result = model.predict(pd.DataFrame({"ds": future_dates}))
    return np.clip(result["yhat"].to_numpy(dtype=float), 0, None)

def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    absolute_error = np.abs(actual - predicted)

    actual_total = actual.sum()
    wape = (
        absolute_error.sum() / actual_total * 100
        if actual_total > 0 else np.nan
    )

    positive_actual = actual > 0
    mape_positive = (
        np.mean(
            absolute_error[positive_actual] / actual[positive_actual]
        ) * 100
        if positive_actual.any() else np.nan
    )

    denominator = np.abs(actual) + np.abs(predicted)
    smape_terms = np.divide(
        2 * absolute_error,
        denominator,
        out=np.zeros_like(absolute_error),
        where=denominator != 0
    )

    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "WAPE_Percent": wape,
        "MAPE_Positive_Actual_Days_Percent": mape_positive,
        "SMAPE_Percent": smape_terms.mean() * 100,
        "Actual_Total_Revenue": actual_total,
        "Forecast_Total_Revenue": predicted.sum()
    }

def make_models():
    return {
        "Random Forest": RandomForestRegressor(
            n_estimators=300,
            max_depth=10,
            min_samples_leaf=5,
            max_features=0.8,
            random_state=42,
            n_jobs=-1
        ),
        "Extra Trees": ExtraTreesRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            max_features=0.8,
            random_state=42,
            n_jobs=-1
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42
        )
    }

# --------------------------------------------------
# 1. Fit on early history and choose by validation WAPE
# --------------------------------------------------
early_series = series.loc[:fit_end]
X_early, y_early = build_training_data(series, fit_end)
actual_validation = validation["Daily_Revenue"].to_numpy(dtype=float)

validation_predictions = {
    "Weekly Seasonal Naive": weekly_seasonal_forecast(
        early_series, validation_dates
    ),
    "Prophet - No Yearly Seasonality": prophet_forecast(
        early_series, validation_dates
    )
}

for name, model in make_models().items():
    print(f"Training {name} for validation...")
    model.fit(X_early, np.log1p(y_early))
    validation_predictions[name] = recursive_ml_forecast(
        model, early_series, validation_dates
    )
    del model
    gc.collect()

validation_rows = []
for name, predicted in validation_predictions.items():
    validation_rows.append({
        "Model": name,
        **calculate_metrics(actual_validation, predicted)
    })

validation_results = pd.DataFrame(validation_rows).sort_values(
    "WAPE_Percent"
).reset_index(drop=True)

# Model selection uses validation WAPE only.
best_model_name = validation_results.iloc[0]["Model"]
print(f"\nSelected using validation WAPE: {best_model_name}")

# --------------------------------------------------
# 2. Refit each candidate using the complete train period,
# then evaluate on the untouched final 30 days.
# --------------------------------------------------
full_train_series = series.loc[:train_end]
X_full, y_full = build_training_data(series, train_end)
actual_test = test["Daily_Revenue"].to_numpy(dtype=float)

test_predictions = {
    "Weekly Seasonal Naive": weekly_seasonal_forecast(
        full_train_series, test_dates
    ),
    "Prophet - No Yearly Seasonality": prophet_forecast(
        full_train_series, test_dates
    )
}

for name, model in make_models().items():
    print(f"Training {name} for final test evaluation...")
    model.fit(X_full, np.log1p(y_full))
    test_predictions[name] = recursive_ml_forecast(
        model, full_train_series, test_dates
    )
    del model
    gc.collect()

test_rows = []
for name, predicted in test_predictions.items():
    test_rows.append({
        "Model": name,
        "Selected_By_Validation": name == best_model_name,
        **calculate_metrics(actual_test, predicted)
    })

test_results = pd.DataFrame(test_rows).sort_values(
    "WAPE_Percent"
).reset_index(drop=True)

# Preserve the original actuals and all model predictions for auditability.
predictions_output = pd.DataFrame({
    "Date": test_dates,
    "Actual_Revenue": actual_test
})

for name, predicted in test_predictions.items():
    predictions_output[f"{name}_Forecast"] = predicted

validation_path = REPORT_DIR / "lag_calendar_validation_metrics.csv"
test_metrics_path = REPORT_DIR / "lag_calendar_test_metrics.csv"
predictions_path = REPORT_DIR / "lag_calendar_test_predictions.csv"

validation_results.to_csv(validation_path, index=False)
test_results.to_csv(test_metrics_path, index=False)
predictions_output.to_csv(predictions_path, index=False)

print("\n" + "=" * 70)
print("RETAILPULSE - LAG AND CALENDAR FORECAST COMPARISON")
print("=" * 70)

print("\nValidation metrics (used for model selection):")
display(validation_results.round(3))

print("\nFinal test metrics:")
display(test_results.round(3))

print("\nModel selected by validation:", best_model_name)
print("\nSaved files:")
print(validation_path)
print(test_metrics_path)
print(predictions_path)


In [ ]:

from pathlib import Path
import gc
import numpy as np
import pandas as pd

from prophet import Prophet
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --------------------------------------------------
# 1. Paths and data
# --------------------------------------------------
PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "forecasting"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FULL_PATH = PROCESSED_DIR / "daily_sales_timeseries.csv"
TRAIN_PATH = PROCESSED_DIR / "forecast_train_30day.csv"

daily = pd.read_csv(FULL_PATH, parse_dates=["Date"]).sort_values("Date")
train = pd.read_csv(TRAIN_PATH, parse_dates=["Date"]).sort_values("Date")

series = daily.set_index("Date")["Daily_Revenue"].astype(float)
training_dates = train["Date"].reset_index(drop=True)

FOLD_DAYS = 30
N_FOLDS = 3
LAGS = [1, 2, 3, 7, 14, 21, 28, 35, 56]
ROLLING_WINDOWS = [7, 14, 28, 56]

initial_train_size = len(train) - FOLD_DAYS * N_FOLDS
if initial_train_size <= max(LAGS):
    raise ValueError("Not enough training history for the selected validation design.")

print(f"Training-only records: {len(train)}")
print(f"Initial training window: {initial_train_size} days")
print(f"Validation folds: {N_FOLDS}, each {FOLD_DAYS} days")
print("The final 30-day period is not used in this experiment.")

# --------------------------------------------------
# 2. Feature engineering using past data only
# --------------------------------------------------
def is_black_friday(date):
    november_first = pd.Timestamp(year=date.year, month=11, day=1)
    first_thursday = november_first + pd.Timedelta(
        days=(3 - november_first.weekday()) % 7
    )
    thanksgiving = first_thursday + pd.Timedelta(days=21)
    return int(date.normalize() == (thanksgiving + pd.Timedelta(days=1)).normalize())

def make_features(date, history):
    date = pd.Timestamp(date).normalize()
    features = {}

    for lag in LAGS:
        features[f"lag_{lag}"] = float(
            history.get(date - pd.Timedelta(days=lag), 0.0)
        )

    for window in ROLLING_WINDOWS:
        previous_values = np.array([
            history.get(date - pd.Timedelta(days=i), 0.0)
            for i in range(1, window + 1)
        ], dtype=float)
        features[f"rolling_mean_{window}"] = previous_values.mean()
        features[f"rolling_std_{window}"] = previous_values.std()

    features["day_of_week"] = date.dayofweek
    features["day_of_month"] = date.day
    features["month"] = date.month
    features["is_weekend"] = int(date.dayofweek >= 5)
    features["is_month_start"] = int(date.is_month_start)
    features["is_month_end"] = int(date.is_month_end)
    features["is_december"] = int(date.month == 12)
    features["is_november"] = int(date.month == 11)
    features["is_black_friday"] = is_black_friday(date)

    year_length = 366 if date.is_leap_year else 365
    features["year_sin"] = np.sin(2 * np.pi * date.dayofyear / year_length)
    features["year_cos"] = np.cos(2 * np.pi * date.dayofyear / year_length)

    return features

def build_training_data(history_series, cutoff):
    available = history_series.loc[:cutoff]
    history = {pd.Timestamp(d): float(v) for d, v in available.items()}

    target_dates = available.index[len(LAGS):]
    X = pd.DataFrame([make_features(d, history) for d in target_dates])
    y = available.loc[target_dates].to_numpy(dtype=float)

    if len(X) == 0:
        raise ValueError("No feature rows available for model training.")

    return X, y

def recursive_ml_forecast(model, history_series, future_dates):
    history = {pd.Timestamp(d): float(v) for d, v in history_series.items()}
    predictions = []

    for date in future_dates:
        X_next = pd.DataFrame([make_features(date, history)])
        prediction = max(0.0, float(np.expm1(model.predict(X_next)[0])))
        predictions.append(prediction)
        history[pd.Timestamp(date)] = prediction

    return np.asarray(predictions, dtype=float)

def weekly_seasonal_forecast(history_series, future_dates):
    history = {pd.Timestamp(d): float(v) for d, v in history_series.items()}
    predictions = []

    for date in future_dates:
        date = pd.Timestamp(date)
        prediction = max(
            0.0, history.get(date - pd.Timedelta(days=7), 0.0)
        )
        predictions.append(prediction)
        history[date] = prediction

    return np.asarray(predictions, dtype=float)

def prophet_forecast(history_series, future_dates):
    prophet_train = pd.DataFrame({
        "ds": history_series.index,
        "y": history_series.to_numpy(dtype=float)
    })
    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=True,
        daily_seasonality=False
    )
    model.fit(prophet_train)

    forecast = model.predict(pd.DataFrame({"ds": future_dates}))
    return np.clip(forecast["yhat"].to_numpy(dtype=float), 0, None)

def score_forecast(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    absolute_errors = np.abs(actual - predicted)

    actual_total = actual.sum()
    wape_value = (
        absolute_errors.sum() / actual_total * 100
        if actual_total > 0 else np.nan
    )

    nonzero = actual > 0
    mape_value = (
        np.mean(absolute_errors[nonzero] / actual[nonzero]) * 100
        if nonzero.any() else np.nan
    )

    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "WAPE_Percent": wape_value,
        "MAPE_Positive_Actual_Days_Percent": mape_value
    }

def make_models():
    return {
        "Random Forest": RandomForestRegressor(
            n_estimators=300,
            max_depth=10,
            min_samples_leaf=5,
            max_features=0.8,
            random_state=42,
            n_jobs=-1
        ),
        "Extra Trees": ExtraTreesRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            max_features=0.8,
            random_state=42,
            n_jobs=-1
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42
        )
    }

# --------------------------------------------------
# 3. Run three chronological validation folds
# --------------------------------------------------
fold_rows = []

for fold in range(N_FOLDS):
    start_idx = initial_train_size + fold * FOLD_DAYS
    end_idx = start_idx + FOLD_DAYS

    fold_train = train.iloc[:start_idx].copy()
    fold_validation = train.iloc[start_idx:end_idx].copy()

    if len(fold_validation) != FOLD_DAYS:
        raise ValueError(f"Fold {fold + 1} does not contain {FOLD_DAYS} days.")

    cutoff = fold_train["Date"].max()
    future_dates = fold_validation["Date"].tolist()
    actual = fold_validation["Daily_Revenue"].to_numpy(dtype=float)
    history = series.loc[:cutoff]

    print(
        f"\nFold {fold + 1}: train through {cutoff.date()}, "
        f"validate {future_dates[0].date()} to {future_dates[-1].date()}"
    )

    predictions = {
        "Weekly Seasonal Naive": weekly_seasonal_forecast(history, future_dates),
        "Prophet - No Yearly Seasonality": prophet_forecast(history, future_dates)
    }

    X_fit, y_fit = build_training_data(series, cutoff)

    for name, model in make_models().items():
        print(f"  Fitting {name}...")
        model.fit(X_fit, np.log1p(y_fit))
        predictions[name] = recursive_ml_forecast(
            model, history, future_dates
        )
        del model
        gc.collect()

    for model_name, forecast_values in predictions.items():
        fold_rows.append({
            "Fold": fold + 1,
            "Train_End": cutoff.date().isoformat(),
            "Validation_Start": future_dates[0].date().isoformat(),
            "Validation_End": future_dates[-1].date().isoformat(),
            "Model": model_name,
            **score_forecast(actual, forecast_values)
        })

fold_metrics = pd.DataFrame(fold_rows)

# --------------------------------------------------
# 4. Summarize performance across all validation folds
# --------------------------------------------------
summary = (
    fold_metrics.groupby("Model")
    .agg(
        Mean_WAPE_Percent=("WAPE_Percent", "mean"),
        Std_WAPE_Percent=("WAPE_Percent", "std"),
        Mean_MAPE_Positive_Actual_Days_Percent=(
            "MAPE_Positive_Actual_Days_Percent", "mean"
        ),
        Mean_MAE=("MAE", "mean"),
        Mean_RMSE=("RMSE", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .reset_index()
    .sort_values("Mean_WAPE_Percent")
    .reset_index(drop=True)
)

summary["Selected_By_Mean_Validation_WAPE"] = False
if summary["Mean_WAPE_Percent"].notna().any():
    best_idx = summary["Mean_WAPE_Percent"].idxmin()
    summary.loc[best_idx, "Selected_By_Mean_Validation_WAPE"] = True
    best_model = summary.loc[best_idx, "Model"]
else:
    best_model = None

fold_path = REPORT_DIR / "rolling_origin_validation_fold_metrics.csv"
summary_path = REPORT_DIR / "rolling_origin_validation_summary.csv"

fold_metrics.to_csv(fold_path, index=False)
summary.to_csv(summary_path, index=False)

print("\n" + "=" * 70)
print("RETAILPULSE - ROLLING-ORIGIN VALIDATION RESULTS")
print("=" * 70)

print("\nAverage performance across three validation windows:")
display(summary.round(3))

print("\nPer-fold results:")
display(
    fold_metrics.sort_values(["Fold", "WAPE_Percent"]).round(3)
)

print("\nModel selected by mean validation WAPE:", best_model)

print("\nSaved files:")
print(fold_path)
print(summary_path)



In [ ]:

from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from prophet import Prophet
from sklearn.preprocessing import MinMaxScaler

try:
    import torch
    import torch.nn as nn
except ImportError as exc:
    raise ImportError(
        "PyTorch is required for the LSTM ensemble. "
        "Install it in this notebook's environment, restart the kernel, "
        "and rerun this cell."
    ) from exc

# --------------------------------------------------
# 1. Paths and reproducibility
# --------------------------------------------------
PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "forecasting"
MODEL_DIR = PROJECT_DIR / "models"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "daily_sales_timeseries.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Daily sales file not found: {INPUT_PATH}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch device:", device)

# --------------------------------------------------
# 2. Load and validate the full historical time series
# --------------------------------------------------
daily = pd.read_csv(INPUT_PATH, parse_dates=["Date"])
daily = daily.sort_values("Date").reset_index(drop=True)

if daily["Date"].duplicated().any():
    raise ValueError("Duplicate dates found in the daily time series.")

if daily["Daily_Revenue"].isna().any():
    raise ValueError("Missing revenue values found.")

if (daily["Daily_Revenue"] < 0).any():
    raise ValueError("Negative revenue found.")

expected_dates = pd.date_range(
    daily["Date"].min(),
    daily["Date"].max(),
    freq="D"
)

if not daily["Date"].reset_index(drop=True).equals(
    pd.Series(expected_dates, name="Date")
):
    raise ValueError("The input must contain a continuous daily date sequence.")

history_dates = daily["Date"]
revenue = daily["Daily_Revenue"].to_numpy(dtype=float)

if len(revenue) <= 60:
    raise ValueError("Insufficient history to train the LSTM.")

FORECAST_DAYS = 30
LOOKBACK = 30
EPOCHS = 80

forecast_dates = pd.date_range(
    start=history_dates.max() + pd.Timedelta(days=1),
    periods=FORECAST_DAYS,
    freq="D"
)

# --------------------------------------------------
# 3. Fit the final Prophet model on all historical data
# We disable yearly seasonality because prior experiments
# raised instability warnings with this short history.
# --------------------------------------------------
prophet_train = pd.DataFrame({
    "ds": history_dates,
    "y": revenue
})

prophet_model = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
)

print("\nFitting final Prophet model...")
prophet_model.fit(prophet_train)

prophet_raw = prophet_model.predict(
    pd.DataFrame({"ds": forecast_dates})
)

prophet_prediction = np.clip(
    prophet_raw["yhat"].to_numpy(dtype=float),
    0,
    None
)

prophet_lower = np.clip(
    prophet_raw["yhat_lower"].to_numpy(dtype=float),
    0,
    None
)

prophet_upper = np.clip(
    prophet_raw["yhat_upper"].to_numpy(dtype=float),
    0,
    None
)

# --------------------------------------------------
# 4. Prepare scaled sequences and train final LSTM
# The scaler uses all known history because this is the
# final forward forecast, not a held-out test evaluation.
# --------------------------------------------------
scaler = MinMaxScaler()
scaled_revenue = scaler.fit_transform(
    revenue.reshape(-1, 1)
).astype(np.float32)

X_seq = []
y_seq = []

for i in range(LOOKBACK, len(scaled_revenue)):
    X_seq.append(scaled_revenue[i - LOOKBACK:i])
    y_seq.append(scaled_revenue[i])

X_seq = np.asarray(X_seq, dtype=np.float32)
y_seq = np.asarray(y_seq, dtype=np.float32)

X_tensor = torch.tensor(X_seq, dtype=torch.float32, device=device)
y_tensor = torch.tensor(y_seq, dtype=torch.float32, device=device)

class RevenueLSTM(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.fc(self.dropout(output[:, -1, :]))

lstm_model = RevenueLSTM(hidden_size=32).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.005)

print("\nTraining final LSTM model...")
loss_history = []

for epoch in range(EPOCHS):
    lstm_model.train()
    optimizer.zero_grad()

    predictions = lstm_model(X_tensor)
    loss = criterion(predictions, y_tensor)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        lstm_model.parameters(),
        max_norm=1.0
    )

    optimizer.step()
    loss_history.append(float(loss.item()))

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1}/{EPOCHS} "
            f"- training MSE (scaled): {loss.item():.6f}"
        )

# --------------------------------------------------
# 5. Recursively forecast 30 days with LSTM
# No actual future revenue is used.
# --------------------------------------------------
lstm_model.eval()
rolling_values = scaled_revenue.flatten().tolist()
lstm_scaled_predictions = []

for _ in range(FORECAST_DAYS):
    sequence = np.asarray(
        rolling_values[-LOOKBACK:],
        dtype=np.float32
    ).reshape(1, LOOKBACK, 1)

    input_tensor = torch.tensor(
        sequence,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        next_value = float(lstm_model(input_tensor).item())

    lstm_scaled_predictions.append(next_value)
    rolling_values.append(next_value)

lstm_prediction = scaler.inverse_transform(
    np.asarray(
        lstm_scaled_predictions,
        dtype=np.float32
    ).reshape(-1, 1)
).flatten()

lstm_prediction = np.clip(lstm_prediction, 0, None)

# --------------------------------------------------
# 6. Build the Prophet-LSTM ensemble
# Equal weights are used for transparency. Past tests showed
# this ensemble did not outperform Prophet, so do not assume
# it is the most accurate model.
# --------------------------------------------------
ensemble_prediction = (
    prophet_prediction + lstm_prediction
) / 2

forecast = pd.DataFrame({
    "Date": forecast_dates,
    "Prophet_Forecast_Revenue": prophet_prediction,
    "Prophet_Lower_95": prophet_lower,
    "Prophet_Upper_95": prophet_upper,
    "LSTM_Forecast_Revenue": lstm_prediction,
    "Equal_Weight_Ensemble_Revenue": ensemble_prediction
})

forecast["Forecast_Horizon_Days"] = FORECAST_DAYS
forecast["Training_Data_Last_Date"] = history_dates.max().date().isoformat()
forecast["Forecast_Type"] = (
    "Forward forecast from historical data; "
    "not a current 2026 forecast"
)

# --------------------------------------------------
# 7. Save forecast, model artifacts and summary
# --------------------------------------------------
forecast_path = REPORT_DIR / "retailpulse_final_30day_ensemble_forecast.csv"
summary_path = REPORT_DIR / "retailpulse_final_30day_ensemble_summary.csv"
plot_path = REPORT_DIR / "retailpulse_final_30day_ensemble_forecast.png"
loss_path = REPORT_DIR / "final_lstm_training_history.csv"

forecast.to_csv(forecast_path, index=False)

summary_rows = []
for model_name, column in [
    ("Prophet", "Prophet_Forecast_Revenue"),
    ("LSTM", "LSTM_Forecast_Revenue"),
    ("Equal-Weight Prophet-LSTM Ensemble", "Equal_Weight_Ensemble_Revenue")
]:
    summary_rows.append({
        "Model": model_name,
        "Forecast_Days": len(forecast),
        "Forecast_Start": forecast["Date"].min().date(),
        "Forecast_End": forecast["Date"].max().date(),
        "Forecast_Total_Revenue": forecast[column].sum(),
        "Average_Daily_Forecast_Revenue": forecast[column].mean()
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(summary_path, index=False)

pd.DataFrame({
    "Epoch": range(1, len(loss_history) + 1),
    "Training_MSE_Scaled": loss_history
}).to_csv(loss_path, index=False)

torch.save(
    lstm_model.state_dict(),
    MODEL_DIR / "lstm_revenue_model_final.pt"
)
joblib.dump(
    scaler,
    MODEL_DIR / "lstm_revenue_scaler_final.joblib"
)

# --------------------------------------------------
# 8. Plot historical revenue and all three forecasts
# --------------------------------------------------
history_window = daily.tail(120)

plt.figure(figsize=(14, 6))
plt.plot(
    history_window["Date"],
    history_window["Daily_Revenue"],
    label="Historical Daily Revenue"
)
plt.plot(
    forecast["Date"],
    forecast["Prophet_Forecast_Revenue"],
    label="Prophet Forecast"
)
plt.plot(
    forecast["Date"],
    forecast["LSTM_Forecast_Revenue"],
    label="LSTM Forecast"
)
plt.plot(
    forecast["Date"],
    forecast["Equal_Weight_Ensemble_Revenue"],
    label="Prophet-LSTM Ensemble",
    linewidth=2
)
plt.fill_between(
    forecast["Date"],
    forecast["Prophet_Lower_95"],
    forecast["Prophet_Upper_95"],
    alpha=0.15,
    label="Prophet 95% Prediction Interval"
)
plt.axvline(
    history_dates.max(),
    linestyle="--",
    label="Historical Data Cutoff"
)
plt.title("RetailPulse: Final 30-Day Revenue Forecast")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()

# --------------------------------------------------
# 9. Print verification output
# --------------------------------------------------
print("\n" + "=" * 70)
print("RETAILPULSE - FINAL ENSEMBLE FORECAST")
print("=" * 70)
print(f"Historical data ends: {history_dates.max().date()}")
print(f"Forecast starts: {forecast['Date'].min().date()}")
print(f"Forecast ends: {forecast['Date'].max().date()}")

print("\nForecast summary:")
display(summary.round(2))

print("\nFirst 10 forecast days:")
display(forecast.head(10).round(2))

print("\nSaved files:")
print(forecast_path)
print(summary_path)
print(plot_path)
print(loss_path)
print(MODEL_DIR / "lstm_revenue_model_final.pt")
print(MODEL_DIR / "lstm_revenue_scaler_final.joblib")

print(
    "\nAccuracy note: the project's MAPE <= 12% target has not been met. "
    "This cell generates the final forecast deliverable; it does not "
    "claim improved accuracy."
)
print("The master dataset was not modified.")
